# Experiment 03 — Meaning + Memory + Emotion + Social Context

```
Language comprehension
      ↓
Meaning + memory + emotion + social context   <- this notebook
      ↓
Response planning
      ↓
Word and sentence construction
      ↓
Motor commands
```

Stage 02 produced a *meaning* vector from a sentence in isolation. But a real
listener doesn't respond to meaning alone — the same sentence lands differently
depending on **memory** (what's been said before), **emotion** (the speaker's
apparent emotional state), and **social context** (how formal the relationship is,
how urgent this feels). This stage fuses all four into one representation.

Same scope note as before: **standalone, not wired to stage 02.** We synthesize
stand-in vectors for all four inputs rather than importing stage 02's model.

In [1]:
import random

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
random.seed(0)

## A toy rule linking emotion + social context to response tone

`meaning` (16-dim) and `memory` (8-dim) are just random vectors here — stand-ins for
whatever stages 02 and "conversation history" would actually produce. `emotion` is
one of 5 categories; `social` is 3 continuous features (formality, closeness,
urgency), each in [0, 1].

We need *some* ground truth to train against, so we hand-define a small rule for
what response **tone** fits the situation — deliberately built so it depends only on
emotion and formality, ignoring meaning, memory, closeness, and urgency entirely.
That's on purpose: it lets us later check whether the network actually learned to
attend to the relevant signals, or just memorized noise.

In [2]:
emotions = ["neutral", "happy", "sad", "angry", "anxious"]
tones = ["supportive", "formal", "playful", "urgent"]


def tone_rule(emotion, formality):
    if emotion in ("sad", "anxious"):
        return "supportive"
    if formality > 0.5:
        return "formal"
    if emotion == "happy":
        return "playful"
    return "urgent"


def make_dataset(n):
    meaning, memory, emotion_idx, social, labels = [], [], [], [], []
    for _ in range(n):
        e = random.randrange(len(emotions))
        formality, closeness, urgency = random.random(), random.random(), random.random()
        meaning.append(torch.randn(16))
        memory.append(torch.randn(8))
        emotion_idx.append(e)
        social.append([formality, closeness, urgency])
        labels.append(tones.index(tone_rule(emotions[e], formality)))
    return (
        torch.stack(meaning),
        torch.stack(memory),
        torch.tensor(emotion_idx),
        torch.tensor(social, dtype=torch.float32),
        torch.tensor(labels),
    )


train_meaning, train_memory, train_emotion, train_social, train_labels = make_dataset(200)
test_meaning, test_memory, test_emotion, test_social, test_labels = make_dataset(50)

print("train label counts:", {t: (train_labels == i).sum().item() for i, t in enumerate(tones)})

train label counts: {'supportive': 79, 'formal': 58, 'playful': 21, 'urgent': 42}


## Architecture: project each stream, fuse, classify

Each of the four inputs gets its own small projector into a shared 16-dim space
(`meaning`/`memory` through a `Linear`, `emotion` through an `Embedding` since it's
categorical, `social` through a `Linear`). The four projections are summed and
squashed through `tanh` into the **fused vector** — the thing a downstream stage
would consume. A small classifier head on top predicts `tone`, purely so the fusion
is forced to learn something structured.

In [3]:
class FusionNet(nn.Module):
    def __init__(self, meaning_dim=16, memory_dim=8, n_emotions=5, social_dim=3, fused_dim=16, n_tones=4):
        super().__init__()
        self.meaning_proj = nn.Linear(meaning_dim, fused_dim)
        self.memory_proj = nn.Linear(memory_dim, fused_dim)
        self.emotion_embed = nn.Embedding(n_emotions, fused_dim)
        self.social_proj = nn.Linear(social_dim, fused_dim)
        self.tone_head = nn.Sequential(
            nn.Linear(fused_dim, fused_dim),
            nn.ReLU(),
            nn.Linear(fused_dim, n_tones),
        )

    def forward(self, meaning, memory, emotion_idx, social):
        fused = torch.tanh(
            self.meaning_proj(meaning)
            + self.memory_proj(memory)
            + self.emotion_embed(emotion_idx)
            + self.social_proj(social)
        )
        logits = self.tone_head(fused)
        return logits, fused


model = FusionNet()
logits, fused = model(train_meaning, train_memory, train_emotion, train_social)
print("fused shape:", fused.shape, "logits shape:", logits.shape)

fused shape: torch.Size([200, 16]) logits shape: torch.Size([200, 4])


## Training

Cross-entropy on `tone`, Adam, held-out test set (50 fresh samples the rule generated but the network never trained on).

In [4]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

losses = []
for step in range(400):
    logits, _ = model(train_meaning, train_memory, train_emotion, train_social)
    loss = F.cross_entropy(logits, train_labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

with torch.no_grad():
    train_logits, _ = model(train_meaning, train_memory, train_emotion, train_social)
    train_acc = (train_logits.argmax(dim=1) == train_labels).float().mean().item()
    test_logits, _ = model(test_meaning, test_memory, test_emotion, test_social)
    test_acc = (test_logits.argmax(dim=1) == test_labels).float().mean().item()

print(f"loss: {losses[0]:.3f} -> {losses[-1]:.3f}")
print(f"train accuracy: {train_acc:.2%}   test accuracy: {test_acc:.2%}")

loss: 1.400 -> 0.000
train accuracy: 100.00%   test accuracy: 84.00%


## Checking it learned the *right* thing

High test accuracy just means it learned *a* rule from the 4 inputs — not
necessarily the intended one. Two checks: (1) hand-pick an emotion/social
combination for each rule branch and see if the predicted tone matches; (2) hold
emotion/social fixed and swap in fresh random meaning/memory vectors — since those
are irrelevant to the rule, the prediction should **not** change.

In [5]:
def predict_tone(meaning, memory, emotion_name, formality, closeness, urgency):
    e = torch.tensor([emotions.index(emotion_name)])
    s = torch.tensor([[formality, closeness, urgency]], dtype=torch.float32)
    with torch.no_grad():
        logits, _ = model(meaning, memory, e, s)
    return tones[logits.argmax(dim=1).item()]


fixed_meaning = torch.randn(1, 16)
fixed_memory = torch.randn(1, 8)

print("-- one combination per rule branch (meaning/memory fixed) --")
probe_cases = [
    ("sad", 0.2, "expect supportive"),
    ("neutral", 0.8, "expect formal"),
    ("happy", 0.2, "expect playful"),
    ("angry", 0.1, "expect urgent"),
]
for emotion_name, formality, note in probe_cases:
    predicted = predict_tone(fixed_meaning, fixed_memory, emotion_name, formality, 0.5, 0.5)
    print(f"emotion={emotion_name:8} formality={formality:.1f} -> {predicted:10} ({note})")

print()
print("-- emotion/social fixed at the 'playful' case, meaning/memory swapped 3x --")
for _ in range(3):
    m, mem = torch.randn(1, 16), torch.randn(1, 8)
    predicted = predict_tone(m, mem, "happy", 0.2, 0.5, 0.5)
    print(f"fresh random meaning/memory -> {predicted}")

-- one combination per rule branch (meaning/memory fixed) --
emotion=sad      formality=0.2 -> supportive (expect supportive)
emotion=neutral  formality=0.8 -> formal     (expect formal)
emotion=happy    formality=0.2 -> playful    (expect playful)
emotion=angry    formality=0.1 -> urgent     (expect urgent)

-- emotion/social fixed at the 'playful' case, meaning/memory swapped 3x --
fresh random meaning/memory -> formal
fresh random meaning/memory -> playful
fresh random meaning/memory -> playful


## What this hands off (conceptually)

A single 16-dim `fused` vector per situation, plus a predicted `tone`. Stage 04
(`response planning`) would take a vector shaped like this and decide *what kind of
response to make* — this stage only decided *in what spirit*, not what to actually
say.

Two honest results worth keeping, not smoothing over:
- **Test accuracy landed at 84%, not 100%.** The rule is simple, but formality is a
  continuous value split at a hard 0.5 threshold — points near that boundary are
  genuinely hard to call correctly from noisy training data, so some test misses are
  expected, not a bug.
- **The robustness check wasn't fully robust.** Holding emotion/social fixed at the
  "playful" case and swapping in three fresh random meaning/memory vectors, one of
  the three flipped the prediction to "formal" anyway. Meaning and memory are
  irrelevant *by construction* of the rule, but they still get summed into the same
  fused vector as emotion and social — so their noise can occasionally push a
  near-boundary case over a decision edge. A cleaner design might gate or attend to
  streams by relevance rather than always summing all four; this toy doesn't do
  that, and the flip is the visible cost.